# UELer — Try It on Binder

**Usability Enhanced Linked Viewer** for interactive exploration of spatial proteomics data.

This notebook needs **no local data**. It streams a public [BioImage Archive](https://www.ebi.ac.uk/bioimage-archive/) study directly over the network, so you can explore a real multiplexed imaging dataset in your browser. Just run the cells below.

## The demo study

We open **`S-BIAD2557`** — a MIBI study of murine liver tumours. The `descriptor` tells UELer where the FOV images, segmentation masks and cell table live within the study:

- **`source`**: a BIA accession id (resolved via the BioStudies API) or a direct HTTPS base URL.
- **`descriptor`**: an optional layout map (a `dict` or a path to a `.json`). Omit it to attempt auto-detection.

Each FOV is a `<FOV>.zip` of channel TIFFs, which `fov_container: 'zip'` selects — a single channel is read straight out of the remote archive with a byte-range request. Only the channels you open are streamed/cached; the whole dataset is never downloaded.

## Open the viewer

Run the cell below. The first field of view is fetched on demand.

In [ ]:
import ueler
# Enable the interactive widgets backend
%matplotlib widget

viewer = ueler.run_viewer_bia(
    "S-BIAD2557",
    descriptor={
        "mode": "folder",
        "fov_container": "zip",
        "base": "Files/spatial_murine_iCCAvsHCC/image_data",
        "mask_dir": "Files/spatial_murine_iCCAvsHCC/segmentation/cleaned_mask",
        "mask_glob": "{fov}_*.tiff",
        "cell_table": "Files/spatial_murine_iCCAvsHCC/cell_table/pCSL005_cell_table.csv",
    },
)

## Add the cell table

The study ships a cell table (per-cell area, position, 32 marker intensities, UMAP coordinates and four levels of lineage annotation). Attaching it is what turns on the heatmap, the scatter plot, the cell gallery and expression-driven mask colouring.

The full table is **361 MB / ~440,000 cells across 455 FOVs**, which is more than a free Binder container wants to hold. The cell below therefore keeps only the rows of the first 12 FOVs: the rows are filtered while the file streams, so the slice that lands on disk is ~12 MB (14,344 cells). Reading the remote file end to end still takes **a few minutes** (about five on a slow link) — a CSV has no index, so there is no way to skip ahead to the rows we want.

Pass `fovs=None` for every FOV (~400 MB in memory — fine locally, risky on Binder), or widen the slice to whichever FOVs you plan to open.

In [ ]:
ueler.load_bia_cell_table(viewer, fovs=viewer.available_fovs[:12])

## Where to next

- Try a **different study** by changing `source` (and `descriptor`) above — see `script/run_ueler_BIA.ipynb` for more worked examples.
- Run UELer on **your own local data** with `script/run_ueler.ipynb`.
- Read the [documentation](https://hartmannlab.github.io/UELer/) for the full user guide.